In [0]:
%sql

CREATE or replace TABLE com_edp_prd.cmpa_insights_internal_schema.tableau_ga_dashboard_base_table AS

WITH mktg_webevents AS (
    -- Base events
    SELECT
        eventName AS eventname,
        `customEvent:click_text` AS content,
        TRY_CAST(engagedSessions AS DOUBLE) AS engagedsessions,
        source_page,
        load_date
    FROM com_intgr.mktg_eventname

    UNION ALL

    SELECT
        'bounceRate' AS eventname,
        NULL AS content,
        TRY_CAST(_bounceRate AS DOUBLE) AS engagedsessions,
        source_page,
        load_date
    FROM com_intgr.mktg_bouncerate

    UNION ALL

    SELECT
        'totaluser' AS eventname,
        NULL AS content,
        TRY_CAST(totalUsers AS DOUBLE) AS engagedsessions,
        source_page,
        load_date
    FROM com_intgr.mktg_totalusers
),

/* ---------------------------------------------------
   Page-level base metrics
--------------------------------------------------- */
base_metrics AS (
    SELECT
        source_page,
        load_date,
        MAX(CASE WHEN eventname = 'totaluser' THEN engagedsessions END) AS total_users,
        MAX(
            CASE
                WHEN eventname = 'user_engagement'
                     AND (content IS NULL OR content = '')
                THEN engagedsessions
            END
        ) AS user_engagement
    FROM mktg_webevents
    WHERE eventname IN ('totaluser', 'user_engagement')
      AND engagedsessions IS NOT NULL
    GROUP BY source_page, load_date
),

/* ---------------------------------------------------
   Active users (daily delta from cumulative total_users)
--------------------------------------------------- */
active_users AS (
    SELECT
        source_page,
        load_date,
        total_users,
        user_engagement,
        COALESCE(
            total_users - LAG(total_users)
                OVER (PARTITION BY source_page ORDER BY load_date),
            total_users
        ) AS active_users
    FROM base_metrics
),

/* ---------------------------------------------------
   Daily delta for specific click_text content
   (EXCLUDING Download – handled separately)
--------------------------------------------------- */
specific_content_daily_total AS (
    SELECT
        content,
        source_page,
        load_date,
        SUM(engagedsessions) AS total_daily_engagedsessions
    FROM mktg_webevents
    WHERE content IN (
        'Understanding biomarkers',
        'Resources',
        'GAGs in the brain',
        'Ongoing GAG buildup',
        'Future advancements',
        'FOR US HEALTHCARE PROFESSIONALS',
        'CNS GAGs',
        'Disease manifestations',
        'Future landscape',
        'Key biomarkers',
        'I am a US healthcare professional'
    )
      AND engagedsessions IS NOT NULL
    GROUP BY content, source_page, load_date
),

specific_content_with_lag AS (
    SELECT
        content,
        source_page,
        load_date,
        total_daily_engagedsessions,
        LAG(total_daily_engagedsessions)
            OVER (PARTITION BY source_page, content ORDER BY load_date) AS prev_total
    FROM specific_content_daily_total
),

daily_content_delta AS (
    SELECT
        content,
        source_page,
        load_date,
        COALESCE(
            total_daily_engagedsessions - prev_total,
            total_daily_engagedsessions
        ) AS daily_delta
    FROM specific_content_with_lag
),

/* ---------------------------------------------------
   Download button – daily delta
--------------------------------------------------- */
download_button_daily_total AS (
    SELECT
        source_page,
        load_date,
        SUM(engagedsessions) AS total_download_clicks
    FROM mktg_webevents
    WHERE eventname = 'download'
      AND content = 'Download'
      AND engagedsessions IS NOT NULL
    GROUP BY source_page, load_date
),

download_button_with_lag AS (
    SELECT
        source_page,
        load_date,
        total_download_clicks,
        LAG(total_download_clicks)
            OVER (PARTITION BY source_page ORDER BY load_date) AS prev_download_clicks
    FROM download_button_daily_total
),

d_download AS (
    SELECT
        'd_download' AS eventname,
        'Download Button Click' AS content,
        COALESCE(
            total_download_clicks - prev_download_clicks,
            total_download_clicks
        ) AS engagedsessions,
        source_page,
        load_date
    FROM download_button_with_lag
),

/* ---------------------------------------------------
   Form submission – daily delta
--------------------------------------------------- */
form_submit_daily_total AS (
    SELECT
        source_page,
        load_date,
        SUM(engagedsessions) AS total_form_submits
    FROM mktg_webevents
    WHERE eventname = 'form_submit'
      AND content = '(not set)'
      AND engagedsessions IS NOT NULL
    GROUP BY source_page, load_date
),

form_submit_with_lag AS (
    SELECT
        source_page,
        load_date,
        total_form_submits,
        LAG(total_form_submits)
            OVER (PARTITION BY source_page ORDER BY load_date) AS prev_form_submits
    FROM form_submit_daily_total
),

d_form_submit AS (
    SELECT
        'd_form_submit' AS eventname,
        'Form Submission' AS content,
        COALESCE(
            total_form_submits - prev_form_submits,
            total_form_submits
        ) AS engagedsessions,
        source_page,
        load_date
    FROM form_submit_with_lag
),

/* ---------------------------------------------------
   Final unified dataset
--------------------------------------------------- */
final_webevents AS (
    -- Raw events
    SELECT
        eventname,
        content,
        engagedsessions,
        source_page,
        load_date
    FROM mktg_webevents

    UNION ALL

    -- Active users
    SELECT
        'activeuser',
        NULL,
        active_users,
        source_page,
        load_date
    FROM active_users
    WHERE active_users IS NOT NULL

    UNION ALL

    -- Average engagement time
    SELECT
        'avg_engagement_time',
        NULL,
        CASE
            WHEN active_users > 0 AND user_engagement IS NOT NULL
            THEN user_engagement / active_users
        END,
        source_page,
        load_date
    FROM active_users

    UNION ALL

    -- Daily text click deltas
    SELECT
        CONCAT(
            'daily_',
            LOWER(REPLACE(REPLACE(content, ' ', '_'), '__', '_'))
        ) AS eventname,
        content,
        daily_delta,
        source_page,
        load_date
    FROM daily_content_delta
    WHERE daily_delta IS NOT NULL

    UNION ALL

    -- Download button
    SELECT
        eventname,
        content,
        engagedsessions,
        source_page,
        load_date
    FROM d_download
    WHERE engagedsessions IS NOT NULL

    UNION ALL

    -- Form submit
    SELECT
        eventname,
        content,
        engagedsessions,
        source_page,
        load_date
    FROM d_form_submit
    WHERE engagedsessions IS NOT NULL
)

/* ---------------------------------------------------
   FINAL SELECT (filter safely here)
--------------------------------------------------- */
SELECT
    eventname,
    content,
    engagedsessions,
    source_page,
    load_date
FROM final_webevents
ORDER BY load_date DESC, source_page;

